Imports

In [29]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import yfinance as yf

In [60]:
# =========================================================
# GLOBALS: Change your ticker asset here to test robustness!
# =========================================================
TARGET_TICKER = "AAPL"  # Try changing this to "AAPL", "QQQ", or "NVDA"
START_DATE = "2016-01-01"
END_DATE = "2026-06-20"

print(f"--- Fetching and Engineering Data for: {TARGET_TICKER} ---")
raw_data = yf.download(TARGET_TICKER, start=START_DATE, end=END_DATE)

if isinstance(raw_data.columns, pd.MultiIndex):
    raw_data.columns = raw_data.columns.get_level_values(0)

df = pd.DataFrame(index=raw_data.index)
close = raw_data['Adj Close'] if 'Adj Close' in raw_data.columns else raw_data['Close']
volume = raw_data['Volume']

# =========================================================
# NEW INDICATOR ADDITION: Relative Strength Index (RSI)
# =========================================================
delta = close.diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / (loss + 1e-9)  # Add epsilon to prevent division by zero
df['feat_rsi_14'] = 100 - (100 / (1 + rs))

# =========================================================
# CORE FEATURE ENGINEERING (Includes Multi-Period Growth Indicators)
# =========================================================
# Base Returns
df['feat_return_1d'] = close.pct_change(1)
df['feat_return_5d'] = close.pct_change(5)    # 5-Day Growth Trend
df['feat_return_21d'] = close.pct_change(21)  # 21-Day Growth Trend

# Moving Average Ratios
df['feat_ma_5_ratio'] = (close / close.rolling(5).mean()) - 1
df['feat_ma_21_ratio'] = (close / close.rolling(21).mean()) - 1

# Volatility and Volume Force
df['feat_volatility_21d'] = df['feat_return_1d'].rolling(21).std()
df['feat_volume_ratio'] = (volume / volume.rolling(5).mean()) - 1

# =========================================================
# TARGET DESIGN (Regressor & Classifier)
# =========================================================
df['target_next_day_price'] = close.shift(-1) / close - 1  # 1-day forward % return
df['target_return_1wk'] = (close.shift(-5) - close) / close
df['target_action'] = (df['target_return_1wk'] > 0.005).astype(int)

# Clean up rows impacted by the look-back (RSI/MA) or look-forward (Targets)
df = df.dropna()

print("--- Feature Expansion Layer Complete ---")
print(f"Total structured trading days for {TARGET_TICKER}: {df.shape[0]}")
print(f"Total unique features extracted (X Matrix): {len([c for c in df.columns if c.startswith('feat_')])}")
print("\nVerifying Extended Input Sample:")
df[[c for c in df.columns if c.startswith('feat_')]].head()

--- Fetching and Engineering Data for: AAPL ---


[*********************100%***********************]  1 of 1 completed

--- Feature Expansion Layer Complete ---
Total structured trading days for AAPL: 2604
Total unique features extracted (X Matrix): 8

Verifying Extended Input Sample:


,feat_rsi_14,feat_return_1d,feat_return_5d,feat_return_21d,feat_ma_5_ratio,feat_ma_21_ratio,feat_volatility_21d,feat_volume_ratio
Date,,,,,,,,
2016-02-03,48.174224,0.019793,0.031364,-0.085429,0.006393,-0.013990,0.026860,-0.059497
2016-02-04,45.583469,0.008034,0.032247,-0.054385,0.008089,-0.003356,0.026525,-0.011885
2016-02-05,45.243312,-0.026708,-0.028866,-0.061269,-0.013079,-0.027041,0.026802,0.068779
2016-02-08,47.963500,0.010530,-0.009379,-0.009585,-0.000800,-0.016350,0.025342,0.173191
2016-02-09,47.684924,-0.000210,0.010853,-0.015002,-0.003148,-0.015855,0.025311,-0.065558


Data Loading & Feature Engineering 

Feature 1 (feat_return_1d): Daily Return
Feature 2 (feat_return_5d): 1-Week Return
Feature 3 (feat_return_21d): 1-Month Return
Feature 4 (feat_ma_5_ratio): Fast MA Ratio (5-Day SMA deviation)
Feature 5 (feat_ma_21_ratio): Slow MA Ratio (21-Day SMA deviation)
Feature 6 (feat_volatility_21d): 21-Day Volatility
Feature 7 (feat_volume_ratio): Volume Force Ratio

In [47]:
# 1. Separate features from targets
feature_cols = [c for c in df.columns if c.startswith('feat_')]
X_raw = df[feature_cols].values
y_regressor_raw = df['target_next_day_price'].values  
y_classifier_raw = df['target_action'].values

# 2. Compute Chronological Split Index (80% Train, 20% Test)
split_idx = int(len(df) * 0.80)

X_train_raw, X_test_raw = X_raw[:split_idx], X_raw[split_idx:]
y_reg_train_raw, y_reg_test_raw = y_regressor_raw[:split_idx], y_regressor_raw[split_idx:]
y_clf_train, y_clf_test = y_classifier_raw[:split_idx], y_classifier_raw[split_idx:]

# 3. Fit Scaler ONLY on Training Data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# 4. Convert all sets into PyTorch Float Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_reg_train_tensor = torch.tensor(y_reg_train_raw, dtype=torch.float32).unsqueeze(1)
y_reg_test_tensor = torch.tensor(y_reg_test_raw, dtype=torch.float32).unsqueeze(1)
y_clf_train_tensor = torch.tensor(y_clf_train, dtype=torch.float32).unsqueeze(1)
y_clf_test_tensor = torch.tensor(y_clf_test, dtype=torch.float32).unsqueeze(1)

print("--- Cell 3 Complete ---")
print(f"Training days: {X_train_tensor.shape[0]} | Testing days: {X_test_tensor.shape[0]}")

--- Cell 3 Complete ---
Training days: 2083 | Testing days: 521


Chronological Train/Test Split & Feature Scaling/Tensor Setting

In [48]:
class MarketDataset(Dataset):
    def __init__(self, features, reg_targets, clf_targets):
        self.features = features
        self.reg_targets = reg_targets
        self.clf_targets = clf_targets
        
    def __len__(self): return len(self.features)
    def __getitem__(self, idx): return self.features[idx], self.reg_targets[idx], self.clf_targets[idx]

BATCH_SIZE = 32
train_loader = DataLoader(MarketDataset(X_train_tensor, y_reg_train_tensor, y_clf_train_tensor), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(MarketDataset(X_test_tensor, y_reg_test_tensor, y_clf_test_tensor), batch_size=BATCH_SIZE, shuffle=False)

print("--- Cell 4 Complete: DataLoaders Ready ---")

--- Cell 4 Complete: DataLoaders Ready ---


PyTorch Dataset & DataLoader Setup

In [49]:
class MarketRegressor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1)
        )
    def forward(self, x): return self.network(x)

class MarketClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1)
        )
    def forward(self, x): return self.network(x)

INPUT_DIMENSIONS = X_train_tensor.shape[1]
reg_model = MarketRegressor(input_dim=INPUT_DIMENSIONS)
clf_model = MarketClassifier(input_dim=INPUT_DIMENSIONS)

print("--- Cell 5 Complete: Models Instantiated ---")

--- Cell 5 Complete: Models Instantiated ---


Neural Network Model Architectures

In [50]:
criterion_regressor = nn.MSELoss()
criterion_classifier = nn.BCEWithLogitsLoss()

optimizer_regressor = optim.Adam(reg_model.parameters(), lr=0.001)
optimizer_classifier = optim.Adam(clf_model.parameters(), lr=0.001)

print("--- Cell 6 Complete: Optimizers Bound ---")

--- Cell 6 Complete: Optimizers Bound ---


Optimizers and Loss Functions

In [51]:
import torch.optim as optim

# 1. Define Loss Functions
criterion_regressor = nn.MSELoss()
criterion_classifier = nn.BCEWithLogitsLoss()

# 2. Define Optimizers (Adam handles dynamic learning rates gracefully)
LEARNING_RATE = 0.001

optimizer_regressor = optim.Adam(reg_model.parameters(), lr=LEARNING_RATE)
optimizer_classifier = optim.Adam(clf_model.parameters(), lr=LEARNING_RATE)

print("--- Optimization Engine Ready ---")
print(f"Regressor Loss:  {criterion_regressor.__class__.__name__}")
print(f"Classifier Loss: {criterion_classifier.__class__.__name__}")
print(f"Learning Rate:   {LEARNING_RATE}")

--- Optimization Engine Ready ---
Regressor Loss:  MSELoss
Classifier Loss: BCEWithLogitsLoss
Learning Rate:   0.001


Dual Training Loop

In [52]:
EPOCHS = 20
print("--- Starting Model Training ---")

for epoch in range(1, EPOCHS + 1):
    reg_model.train(); clf_model.train()
    running_reg_loss, running_clf_loss = 0.0, 0.0
    
    for batch_features, batch_reg_targets, batch_clf_targets in train_loader:
        # Regressor Update
        optimizer_regressor.zero_grad()
        loss_reg = criterion_regressor(reg_model(batch_features), batch_reg_targets)
        loss_reg.backward(); optimizer_regressor.step()
        running_reg_loss += loss_reg.item() * batch_features.size(0)
        
        # Classifier Update
        optimizer_classifier.zero_grad()
        loss_clf = criterion_classifier(clf_model(batch_features), batch_clf_targets)
        loss_clf.backward(); optimizer_classifier.step()
        running_clf_loss += loss_clf.item() * batch_features.size(0)
        
    if epoch == 1 or epoch % 2 == 0:
        print(f"Epoch {epoch:02d}/{EPOCHS} | Regressor Loss: {running_reg_loss/len(X_train_tensor):.6f} | Classifier Loss: {running_clf_loss/len(X_train_tensor):.4f}")

--- Starting Model Training ---
Epoch 01/20 | Regressor Loss: 0.001771 | Classifier Loss: 0.6937
Epoch 02/20 | Regressor Loss: 0.000669 | Classifier Loss: 0.6885
Epoch 04/20 | Regressor Loss: 0.000507 | Classifier Loss: 0.6864
Epoch 06/20 | Regressor Loss: 0.000584 | Classifier Loss: 0.6840
Epoch 08/20 | Regressor Loss: 0.000328 | Classifier Loss: 0.6837
Epoch 10/20 | Regressor Loss: 0.000188 | Classifier Loss: 0.6834
Epoch 12/20 | Regressor Loss: 0.000159 | Classifier Loss: 0.6801
Epoch 14/20 | Regressor Loss: 0.000155 | Classifier Loss: 0.6767
Epoch 16/20 | Regressor Loss: 0.000149 | Classifier Loss: 0.6778
Epoch 18/20 | Regressor Loss: 0.000156 | Classifier Loss: 0.6773
Epoch 20/20 | Regressor Loss: 0.000146 | Classifier Loss: 0.6745


Out-of-Sample Evaluation

In [53]:
reg_model.eval(); clf_model.eval()

with torch.no_grad():
    test_reg_preds = reg_model(X_test_tensor).numpy()
    test_clf_logits = clf_model(X_test_tensor)
    test_clf_preds = (test_clf_logits >= 0.0).float().numpy()

actual_returns = y_reg_test_tensor.numpy()
actual_actions = y_clf_test_tensor.numpy()

mae_pct = np.mean(np.abs(actual_returns - test_reg_preds)) * 100
rmse_pct = np.sqrt(np.mean((actual_returns - test_reg_preds) ** 2)) * 100
accuracy = accuracy_score(actual_actions, test_clf_preds)

print("==================================================")
print("          SYNCHRONIZED EVALUATION RESULTS         ")
print("==================================================")
print(f"--- REGRESSOR (Next-Day % Return Prediction) ---")
print(f"Mean Absolute Error (MAE):  {mae_pct:.4f}%")
print(f"Root Mean Squared Error (RMSE): {rmse_pct:.4f}%")
print(f"\n--- CLASSIFIER (1-Week Trend Binary Signal) ---")
print(f"Directional Prediction Accuracy: {accuracy * 100:.2f}%")
print("\nDetailed Classification Breakdown:")
print(classification_report(actual_actions, test_clf_preds, target_names=["Hold/Sell (0)", "Buy Signal (1)"]))

          SYNCHRONIZED EVALUATION RESULTS         
--- REGRESSOR (Next-Day % Return Prediction) ---
Mean Absolute Error (MAE):  0.7092%
Root Mean Squared Error (RMSE): 1.0478%

--- CLASSIFIER (1-Week Trend Binary Signal) ---
Directional Prediction Accuracy: 53.36%

Detailed Classification Breakdown:
                precision    recall  f1-score   support

 Hold/Sell (0)       0.53      0.50      0.51       257
Buy Signal (1)       0.54      0.57      0.55       264

      accuracy                           0.53       521
     macro avg       0.53      0.53      0.53       521
  weighted avg       0.53      0.53      0.53       521



Testing Suite

In [61]:
# =========================================================
# TEST ENGINE: Input any ticker to test out-of-sample models
# =========================================================
TEST_TICKER = "QQQ"  # Try changing this to "NVDA", "MSFT", "SPY", etc.

print(f"--- Running Sandbox Prediction Engine on: {TEST_TICKER} ---")
test_raw = yf.download(TEST_TICKER, start="2016-01-01", end="2026-06-20")

if isinstance(test_raw.columns, pd.MultiIndex):
    test_raw.columns = test_raw.columns.get_level_values(0)

# Build features matching your exact Cell 2 logic (8 features total)
test_df = pd.DataFrame(index=test_raw.index)
t_close = test_raw['Adj Close'] if 'Adj Close' in test_raw.columns else test_raw['Close']
t_volume = test_raw['Volume']

# 1. Calculate RSI
delta = t_close.diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / (loss + 1e-9)
test_df['feat_rsi_14'] = 100 - (100 / (1 + rs))

# 2. Core Features
test_df['feat_return_1d'] = t_close.pct_change(1)
test_df['feat_return_5d'] = t_close.pct_change(5)
test_df['feat_return_21d'] = t_close.pct_change(21)
test_df['feat_ma_5_ratio'] = (t_close / t_close.rolling(5).mean()) - 1
test_df['feat_ma_21_ratio'] = (t_close / t_close.rolling(21).mean()) - 1
test_df['feat_volatility_21d'] = test_df['feat_return_1d'].rolling(21).std()
test_df['feat_volume_ratio'] = (t_volume / t_volume.rolling(5).mean()) - 1

# 3. Target Setup
test_df['target_next_day_price'] = t_close.shift(-1) / t_close - 1
test_df['target_return_1wk'] = (t_close.shift(-5) - t_close) / t_close
test_df['target_action'] = (test_df['target_return_1wk'] > 0.005).astype(int)
test_df = test_df.dropna()

# Isolate features
sandbox_feature_cols = [c for c in test_df.columns if c.startswith('feat_')]
X_ticker_raw = test_df[sandbox_feature_cols].values

# Final safety check before processing
if X_ticker_raw.shape[1] != scaler.n_features_in_:
    print(f"\n❌ Shape Mismatch: Sandbox generated {X_ticker_raw.shape[1]} features, but Scaler expects {scaler.n_features_in_}.")
    print("👉 QUICK FIX: Make sure you run Cell 3 through Cell 5 sequentially to lock in the 8-feature layout.")
else:
    # Scale and process tensors
    X_ticker_scaled = scaler.transform(X_ticker_raw)
    X_ticker_tensor = torch.tensor(X_ticker_scaled, dtype=torch.float32)

    # Put models in evaluation mode and predict
    reg_model.eval()
    clf_model.eval()
    with torch.no_grad():
        ticker_reg_preds = reg_model(X_ticker_tensor).numpy()
        ticker_clf_logits = clf_model(X_ticker_tensor)
        ticker_clf_preds = (ticker_clf_logits >= 0.0).float().numpy()

    # Calculate Performance Metrics on this new asset
    act_ret = test_df['target_next_day_price'].values
    act_act = test_df['target_action'].values

    t_mae = np.mean(np.abs(act_ret - ticker_reg_preds.flatten())) * 100
    t_acc = accuracy_score(act_act, ticker_clf_preds.flatten())

    print("==================================================")
    print(f"      SANDBOX RESULTS FOR ASSET: {TEST_TICKER}     ")
    print("==================================================")
    print(f"Regressor Next-Day Error (MAE):     {t_mae:.4f}%")
    print(f"Classifier Move Accuracy:           {t_acc * 100:.2f}%")
    print("==================================================")

    # Show the model's predictions for the last 5 days
    print("\nMost Recent Days Prediction Signals:")
    latest_days = test_df.index[-5:]
    for i, day in enumerate(latest_days):
        pred_move = ticker_reg_preds[-5+i][0] * 100
        signal = "BUY" if ticker_clf_preds[-5+i][0] == 1 else "HOLD/SELL"
        print(f"Date: {day.strftime('%Y-%m-%d')} | Est. Next-Day Return: {pred_move:+.2f}% | 1-Wk Action: {signal}")

[*********************100%***********************]  1 of 1 completed

--- Running Sandbox Prediction Engine on: QQQ ---
      SANDBOX RESULTS FOR ASSET: QQQ     
Regressor Next-Day Error (MAE):     0.9779%
Classifier Move Accuracy:           52.61%

Most Recent Days Prediction Signals:
Date: 2026-06-05 | Est. Next-Day Return: +0.24% | 1-Wk Action: HOLD/SELL
Date: 2026-06-08 | Est. Next-Day Return: +0.13% | 1-Wk Action: BUY
Date: 2026-06-09 | Est. Next-Day Return: -0.38% | 1-Wk Action: HOLD/SELL
Date: 2026-06-10 | Est. Next-Day Return: +0.29% | 1-Wk Action: HOLD/SELL
Date: 2026-06-11 | Est. Next-Day Return: -0.16% | 1-Wk Action: HOLD/SELL
